# 04 — Explainability + Clinical Error Analysis (Phase 5)

Grad-CAM (via `pytorch-grad-cam`) on the last conv block (`model.layer4[-1]`) of the trained baseline, reviewed on 5 correct + 5 incorrect test-set predictions.

In [ ]:
import sys, json
sys.path.insert(0, "..")
import torch
from torch.utils.data import DataLoader

from src.data_utils import load_dataset_metadata, train_val_test_split
from src.train import HYGDDataset, build_model
from src.evaluate import evaluate
from src.visualize import grad_cam_overlay, review_predictions

df = load_dataset_metadata("../data/raw")
train_df, val_df, test_df = train_val_test_split(df)
test_ds = HYGDDataset(test_df)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

model = build_model(num_classes=2, freeze_backbone=True)
model.load_state_dict(torch.load("../results/baseline_resnet18.pt", map_location="cpu"))
model.eval()

metrics = evaluate(model, test_loader, device="cpu")
sample = review_predictions(metrics["y_true"], metrics["y_prob"], n_correct=5, n_wrong=5)
sample

## Correct predictions

![correct](../figures/08_gradcam_correct.png)

The heatmap concentrates on the optic disc / peripapillary region in most cases — clinically the right area to look at for glaucomatous cupping, which is reassuring (the model isn't just picking up on unrelated image artifacts).

## Incorrect predictions — clinical error analysis

The best model makes 5 errors on the 44-patient test set: **3 false negatives** (missed glaucoma) and **2 false positives** (false alarms). See `../figures/16_the_5_errors_annotated.png` for each original paired with its Grad-CAM. Two distinct failure modes emerge.

### Failure mode A — localization (the model didn't look at the disc)

In 2 of the 3 misses the Grad-CAM heat is off the optic disc entirely:

- **128_0** (P=0.00, quality **2.38 — lowest tier in the set**): markedly underexposed and out of focus; the disc is barely resolvable. This is an *image-quality* failure, not a clinical disagreement — an argument for a **quality gate** (reject/repeat below ~FundusQ 3) rather than something to fix in the classifier.
- **91_3** (P=0.01, quality 6.6): the image is adequate, but Grad-CAM attends to the upper-left, *away* from the disc (lower-right). The model missed the target region, so this is an **attention/detection** failure — an argument for an explicit disc-detection/cropping step before classification.

### Failure mode B — interpretation (right place, wrong call)

Both false positives have Grad-CAM correctly centered on the disc:

- **200_3** (P=0.88, quality 7.3): a clean image whose disc *appears* relatively large and pale with a visible cup and some peripapillary atrophy — features that can mimic glaucomatous cupping. A plausible reading is a **large physiological cup** that the gold-standard work-up (VF/OCT/follow-up) classified as non-glaucomatous. The model over-called on disc morphology alone — exactly the ambiguity that makes fundus-only labels weaker than HYGD's multimodal ones.
- **218_0** (P=0.96, quality 4.7): a hazy image with numerous bright drusen-/exudate-like spots and a pale disc; co-existing findings plus a pale disc plausibly pushed the model toward a false glaucoma call.

### The most instructive case — 91_2 (missed, P=0.47, quality 6.5)

Here the disc actually *looks* glaucomatous to the eye — a large, pale cup with visible laminar dots — yet the model sat right on the fence (P=0.47, a near-miss, not a confident error). Together with **91_3 (same patient, also missed)**, this suggests the difficulty is **patient/disc-specific, not random**: the model has not fully learned the "large excavated cup → glaucoma" pattern a clinician reads at a glance. More such discs in training, or disc-centric cropping, would likely help.

### Take-home

The errors are not one problem but two: **localization** failures (fixable *upstream* with quality gating + disc detection) and **interpretation** failures (fixable with more/harder training discs). And they are **not** explained by image quality alone (Mann-Whitney p = 0.45 across all 5), so the interpretation cases are genuine hard examples, not just bad photos.

> **Caveat on the clinical reading.** The disc descriptions above are observational impressions to *explain model behaviour*, made without slit-lamp / OCT / visual-field context. They are hypotheses for why the model failed, not diagnoses. HYGD's labels come from a full ophthalmic work-up — which is exactly why they are the reference and these single-image impressions are not.

## Limitations recap (see README.md for the full list)

- Single dataset (HYGD), single hospital (Hillel Yaffe Medical Center) — no external validation.
- Small test set: 99 images / 44 patients.
- Baseline only (frozen backbone, 8 epochs) — not tuned for maximum performance.
- **This is a student portfolio/research artifact, not a clinical device.**